# Andrej Karpathy
Credit: Let's build the GPT Tokenizer, https://www.youtube.com/watch?v=zduSFxRajkE

In [1]:
text = "my name is brian"
print(text)
ord("h") # the UNICODE value (integer) for the underlying character

"""
A string in python is simply a sequence on Unicode code points which are defined by the Unicode standard of encoding.

We can not use these UNICODE values because, here we're at a character level and the attention in the transformer
model is going to go out of context due to the limited context length. This means that each character will be represented
by its corrensponding value and you can have so many of these values such that when the attention layer needs to go back 
to all the tokens which are currently being processed - within the context length - the context will either make no sense or will be lost

"""

my name is brian


"\nA string in python is simply a sequence on Unicode code points which are defined by the Unicode standard of encoding.\n\nWe can not use these UNICODE values because, here we're at a character level and the attention in the transformer\nmodel is going to go out of context due to the limited context length. This means that each character will be represented\nby its corrensponding value and you can have so many of these values such that when the attention layer needs to go back \nto all the tokens which are currently being processed - within the context length - the context will either make no sense or will be lost\n\n"

In [4]:
print(chr(104)) # given the UNICODE value, we want to return the character
[print(x, ord(x)) for x in text]

h
m 109
y 121
  32
n 110
a 97
m 109
e 101
  32
i 105
s 115
  32
b 98
r 114
i 105
a 97
n 110


[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]

In [ ]:
""" 
Alternatively, we can encode the string to get the raw bytes which can be used but this approach also suffers from
the problem stated above;
"""

# The encoding can be utf-8, utf-16, or utf-32 (each one has its limitations in this process)
text.encode(encoding='utf-16')

b'\xff\xfem\x00y\x00 \x00n\x00a\x00m\x00e\x00 \x00i\x00s\x00 \x00b\x00r\x00i\x00a\x00n\x00'

## Implementing BPE

Credit: https://huggingface.co/learn/nlp-course/chapter6/5?fw=pt

In [1]:
corpus = [
    "This is the Hugging Face Course.",
    "This chapter is about tokenization.",
    "This section shows several tokenizer algorithms.",
    "Hopefully, you will be able to understand how they are trained and generate tokens.",
]

In [ ]:
from transformers import AutoTokenizer
from collections import defaultdict

tokenizer = AutoTokenizer.from_pretrained("gpt2")

In [ ]:
# compute the frequencies of each word in the corpus as we do the pre-tokenization

word_freqs = defaultdict(int)
for text in corpus:
    # the pre-tokenization step
    words_with_offsets = tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str(text)
    new_words = [word for word, _ in words_with_offsets]
    for word in new_words:
        word_freqs[word] += 1
print(word_freqs)

In [ ]:
# Compute the base vocabulary

alphabet = []
for word in word_freqs.keys():
    for letter in word:
        if letter not in alphabet:
            alphabet.append(letter)
alphabet.sort()
print(alphabet)

In [ ]:
# We also add the special tokens used by the model at the beginning of that vocabulary. In the case of GPT-2, the only special token is "<|endoftext|>"
vocab = ["<|endoftext|>"] + alphabet.copy()

In [ ]:
# split each word into individual letters
splits = {word: [c for c in word] for word in word_freqs.keys()}

In [ ]:
def compute_pair_freqs(splits):
    pair_freqs = defaultdict(int)
    for word, freq in word_freqs.items():
        split = splits[word]
        if len(split) == 1:
            continue
        for i in range(len(split) - 1):
            pair = (split[i], split[i + 1])
            pair_freqs[pair] += freq
    return pair_freqs

## Building a Tokenizer - block-by-block
Credit: <a href='https://huggingface.co/learn/nlp-course/chapter6/8#building-a-bpe-tokenizer-from-scratch'>HuggingFace</a>